Step 1 - Imports and Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog novacart_workspace_adb")
spark.sql("Create schema if not exists silver_schema")

silver_run_id = str(uuid.uuid4())
print("Current Silver Run ID: ", silver_run_id)

In [0]:
spark.sql("""
          create table if not exists novacart_workspace_adb.silver_schema.processing_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_prosessed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          """)

In [0]:
def upsert_to_silver(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
         .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)


In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl = (spark.table("novacart_workspace_adb.silver_schema.      processing_control")
        .filter(
            (F.col("layer") == "silver") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "SUCCESS")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None, None

    return rows[0]["last_processed_bronze_ingested_at"], rows[0]["last_processed_bronze_run_id"]

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "SUCCESS",
                silver_run_id,
                datetime.now()
            )
        ],
        schema="""
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at timestamp
            """
    )

    dt = DeltaTable.forName(spark, "novacart_workspace_adb.silver_schema.processing_control")
    (dt.alias("t")
        .merge(ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name")
        .whenMatchedUpdate(set={
            "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at": "s.last_processed_bronze_ingested_at",
            "rows_merged": "s.rows_merged",
            "run_status": "s.run_status",
            "silver_run_id": "s.silver_run_id",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute())

    

In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at, last_run_id = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)

    if last_ingested_at is None:
        return bronze_df, last_ingested_at, last_run_id

    return bronze_df.filter(F.col("bronze_ingested_at") > F.lit(last_ingested_at)), last_ingested_at, last_run_id

In [0]:
df_raw = spark.sql("select * from novacart_workspace_adb.bronze_schema.orders_raw")
display(df_raw)

In [0]:
# Step 4 - Orders incremental processing
# Read only the Bronze order rows that Silver has not processed yet.
orders_inc, last_orders_ingested_at, last_orders_run_id = get_incremental_bronze(
    "novacart_workspace_adb.bronze_schema.orders_raw",
    "orders"
)

# Count the incremental order rows entering Silver in this run.
orders_inc_count = orders_inc.count()
print(f"orders rows_to_process_in_silver = {orders_inc_count}")

# Only run Silver order cleaning and validation when there are new Bronze order rows.
if orders_inc_count > 0:
    # Create a window that keeps the latest order record for each order_id.
    order_window = Window.partitionBy("order_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the Silver order-cleaning pipeline. This block standardizes and deduplicates raw order records.
    orders_cleaned = (
        orders_inc
        # Standardize order_status to uppercase so values such as shipped and SHIPPED become consistent.
        .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
        .withColumn("order_status", F.when(F.col("order_status") == "", F.lit(None)).otherwise(F.col("order_status")))
        # Remove formatting characters from order_amount so it can be cast to a numeric type.
        .withColumn("order_amount", F.regexp_replace(F.col("order_amount"), r"[\$, ]", ""))
        .withColumn("order_amount", F.when(F.trim(F.col("order_amount")).isin("N/A", "NULL", "??", ""), None).otherwise(F.col("order_amount")))
        .withColumn("order_amount", F.col("order_amount").cast("double"))
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(order_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_cleaned,
        "novacart_workspace_adb.silver_schema.orders_cleaned",
        "order_id"
    )

    # Apply Silver data-quality rules to the cleaned order records.
    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_by_orders_team",
            F.when(F.col("customer_id").isNull(), "verify_customer_id")
            .when(F.col("product_id").isNull(), "verify_product_id")
            .when(F.col("order_status").isNull() | (F.trim(F.col("order_status")) == ""), "verify_order_status")
            .when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), "verify_order_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_order_amount",
            F.when(F.col("order_amount").isNull() | (F.col("order_amount") <= 0), F.lit(True))
            .otherwise(F.lit(False))
        )
        .withColumn("order_date", F.to_date("created_at"))
        .withColumn("order_year", F.year("created_at"))
        .withColumn("order_month", F.month("created_at"))
        .withColumn("order_day", F.dayofmonth("created_at"))
        .withColumn("order_dow", F.date_format("created_at", "E"))
    )

    # Keep only valid order rows for the transformed Silver table.
    orders_good = orders_validated.filter(F.col("to_be_verified_by_orders_team") == "No Issues")
    
    # Send invalid order rows to the quarantine dataset for manual review.
    orders_bad = (
        orders_validated
        .filter(F.col("to_be_verified_by_orders_team") != "No Issues")
        .withColumn("quarantine_ts", F.current_timestamp())
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(
        orders_good,
        "novacart_workspace_adb.silver_schema.orders_transformed",
        "order_id"
    )

    # Append bad order rows to the quarantine table instead of losing them.
    orders_bad.write.format("delta").mode("append").saveAsTable(
        "novacart_workspace_adb.silver_schema.orders_quarantine"
    )

    # Update control table metadata
    mx_ingested = orders_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = (
        orders_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested))
        .agg(F.max("bronze_run_id").alias("mx"))
        .collect()[0]["mx"]
    )
    
    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())

else:
    print("No new orders Bronze rows for Silver.")
    upsert_silver_control(
        "orders",
        last_orders_run_id,
        last_orders_ingested_at,
        orders_inc_count
    )

In [0]:
%sql
select * from novacart_workspace_adb.silver_schema.orders_quarantine

In [0]:
# Step 5 - Products incremental processing
# Read only the Bronze product rows that Silver has not processed yet.
products_inc, last_products_ingested_at, last_products_run_id = get_incremental_bronze("novacart_workspace_adb.bronze_schema.products_raw", "products")

# Count the incremental product rows entering Silver in this run.
products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver = {products_inc_count}")

if products_inc.count() > 0:
    # Create a window that keeps the latest product record for each product_id.
    product_window = Window.partitionBy("product_id").orderBy(
        F.col("updated_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    # Start the Silver product-cleaning pipeline. This block standardizes and deduplicates raw product records.
    products_cleaned = (
        products_inc
        # Standardize product_name by trimming spaces and converting text to uppercase.
        .withColumn("product_name", F.upper(F.trim(F.col("product_name"))))
        .withColumn("product_name", F.regexp_replace(F.col("product_name"), r"[-_]", " ")) ########### New Logic
        .withColumn("product_name", F.when(F.col("product_name") == "", F.lit(None)).otherwise(F.col("product_name")))
        .withColumn(
            "category",
            F.when(F.upper(F.trim(F.col("category"))).contains("ELECTRNICS"), "ELECTRONICS")
            # .when(F.upper(F.trim(F.col("category"))) == "FITNESS", "FITNESS")
            # .when(F.upper(F.trim(F.col("category"))) == "LIFESTYLE", "LIFESTYLE")
            .otherwise(F.upper(F.trim(F.col("category"))))
        )
        # Start cleaning the product price field before converting it to numeric.
        .withColumn("price", F.trim(F.col("price")))
        .withColumn("price", F.regexp_replace(F.col("price"), r"\$", ""))
        .withColumn("price", F.regexp_replace(F.col("price"), ",", "."))
        .withColumn("price", F.regexp_replace(F.col("price"), r"\s+", ""))
        .withColumn("price", F.expr("try_cast(price as double)"))
        .withColumn("updated_at", F.to_timestamp("updated_at"))
        # Assign a row number inside each business key so we can keep only the latest version of that record.
        .withColumn("row_rank", F.row_number().over(product_window))
        # Keep only the latest record for each business key.
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(products_cleaned, "novacart_workspace_adb.silver_schema.products_cleaned", "product_id")

    # Apply Silver data-quality rules to the cleaned product records.
    products_validated = (
        products_cleaned
        .withColumn(
            "to_be_verified_by_products_team",
            F.when(F.col("product_name").isNull(), "verify_product_name")
            .when(F.col("category").isNull(), "verify_category")
            .when(F.col("price").isNull() | (F.col("price") <= 0), "verify_price")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_product_price",
            F.when(F.col("price").isNull() | (F.col("price") <= 0), "invalid_price").otherwise("valid_price")
        )
    )

    # Keep only valid product rows for the transformed Silver table.
    products_good = products_validated.filter(
        (F.col("to_be_verified_by_products_team") == "No Issues") &
        (F.col("check_product_price") == "valid_price")
    )

    if "price_raw" in products_good.columns:
        # Keep only valid product rows for the transformed Silver table.
        products_good = products_good.drop("price_raw")

    # Send invalid product rows to the quarantine dataset for manual review.
    products_bad = products_validated.filter(
        (F.col("to_be_verified_by_products_team") != "No Issues") |
        (F.col("check_product_price") == "invalid_price")
    ).withColumn("quarantine_ts", F.current_timestamp())

    # Merge the cleaned or validated Silver dataset into its Delta target table.
    upsert_to_silver(products_good, "novacart_workspace_adb.silver_schema.products_transformed", "product_id")
    
    # Append bad product rows to the quarantine table instead of losing them.
    products_bad.write.format("delta").mode("append").saveAsTable("novacart_workspace_adb.silver_schema.products_quarantine")

    mx_ingested = products_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = products_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control("products", mx_run, mx_ingested, products_good.count())

else:
    print("No new products Bronze rows for Silver.")
    upsert_silver_control(
        "products",
        last_products_run_id,
        last_products_ingested_at,
        products_inc_count
    )

In [0]:
%sql
select * from novacart_workspace_adb.silver_schema.products_quarantine

In [0]:
# Step 6 - Payments incremental processing
payments_inc, last_payments_ingested_at, last_payments_run_id = get_incremental_bronze("novacart_workspace_adb.bronze_schema.payments_raw", "payments")
print("Payments last processed Bronze ingested_at =", last_payments_ingested_at)

payments_inc_count = payments_inc.count()
print(f"payments rows_to_process_in_silver = {payments_inc_count}")

if payments_inc.count() > 0:
    payment_window = Window.partitionBy("payment_id").orderBy(
        F.col("processed_at").cast("timestamp").desc(),
        F.col("bronze_ingested_at").desc()
    )

    payments_cleaned = (
        payments_inc
        .withColumn("payment_status", F.upper(F.trim(F.col("payment_status"))))
        .withColumn("payment_status", F.when(F.col("payment_status") == "", F.lit(None)).otherwise(F.col("payment_status")))
        .withColumn("paid_amount", F.trim(F.col("paid_amount")))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r"\$", ""))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), ",", "."))
        .withColumn("paid_amount", F.regexp_replace(F.col("paid_amount"), r"\s+", ""))
        .withColumn("paid_amount", F.expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at", F.to_timestamp("processed_at"))
        .withColumn("row_rank", F.row_number().over(payment_window))
        .filter(F.col("row_rank") == 1)
        .drop("row_rank")
        .withColumn("silver_run_id", F.lit(silver_run_id))
    )

    upsert_to_silver(payments_cleaned, "novacart_workspace_adb.silver_schema.payments_cleaned", "payment_id")

    payments_validated = (
        payments_cleaned
        .withColumn(
            "to_be_verified_by_payments_team",
            F.when(F.col("order_id").isNull(), "verify_order_id")
            .when(F.col("payment_status").isNull(), "verify_payment_status")
            .when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), "verify_paid_amount")
            .otherwise("No Issues")
        )
        .withColumn(
            "check_paid_amount",
            F.when(F.col("paid_amount").isNull() | (F.col("paid_amount") <= 0), F.lit(True)).otherwise(F.lit(False))
        )
    )

    payments_good = payments_validated.filter(F.col("to_be_verified_by_payments_team") == "No Issues")
    
    payments_bad = payments_validated.filter(F.col("to_be_verified_by_payments_team") != "No Issues") \
        .withColumn("quarantine_ts", F.current_timestamp())

    upsert_to_silver(payments_good, "novacart_workspace_adb.silver_schema.payments_transformed", "payment_id")
    payments_bad.write.format("delta").mode("append").saveAsTable("novacart_workspace_adb.silver_schema.payments_quarantine")

    # Fixed mx aggregation placement
    mx_ingested = payments_inc.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = payments_inc.filter(F.col("bronze_ingested_at") == F.lit(mx_ingested)).agg(F.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control("payments", mx_run, mx_ingested, payments_good.count())
else:
    print("No new payments Bronze rows for Silver.")
    upsert_silver_control("payments", last_payments_run_id, last_payments_ingested_at, payments_inc_count)

In [0]:
%sql
select * from novacart_workspace_adb.silver_schema.payments_transformed

In [0]:
# Fixed extraction with corrected syntax
print("Products transformed count :", spark.sql("SELECT COUNT(*) FROM novacart_workspace_adb.silver_schema.products_transformed").collect()[0][0])
print("Orders transformed count   :", spark.sql("SELECT COUNT(*) FROM novacart_workspace_adb.silver_schema.orders_transformed").collect()[0][0])
print("Payments transformed count :", spark.sql("SELECT COUNT(*) FROM novacart_workspace_adb.silver_schema.payments_transformed").collect()[0][0])

display(spark.table("novacart_workspace_adb.silver_schema.processing_control").orderBy("entity_name"))